# Predicting next-month arrears

**Question:** can we predict `arrears_next_month` from what is known on the billing date?

**Design decisions (checklist.md)**
- The model will be used on future months, so the test set is the *latest* rows. Accounts repeat across rows, so the test accounts are also *whole entities* never seen in training: 20% of accounts are held out, and only their rows from the last 4 months form the test set. Held-out accounts' earlier rows are discarded, not used for training.
- `final_notice_sent` is recorded in the cycle *after* the billing month, so it is not known at prediction time (and is near-perfectly tied to the target). It is dropped, as is the identifier `meter_ref`. `arrears_next_month` is the target.
- Cross-validation uses expanding-window time splits inside the training rows; all preprocessing lives inside a Pipeline fit per fold.

In [ ]:
import numpy as np, pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.base import clone
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_score, recall_score
SEED=0
df=pd.read_csv('data.csv',parse_dates=['billing_month'])
print(df.shape, 'positive rate', df.arrears_next_month.mean().round(3))

## 1. Set aside the test set (before any modelling)

In [ ]:
months=np.sort(df.billing_month.unique())
cutoff=months[-4]                       # test period = last 4 months
rng=np.random.default_rng(SEED)
ids=df.meter_ref.unique()
test_ids=set(rng.choice(ids,size=int(0.2*len(ids)),replace=False))
is_test_ent=df.meter_ref.isin(test_ids)
test=df[is_test_ent & (df.billing_month>=cutoff)].copy()
train=df[~is_test_ent & (df.billing_month<cutoff)].copy().sort_values('billing_month').reset_index(drop=True)
assert set(train.meter_ref).isdisjoint(test.meter_ref)
assert train.billing_month.max()<test.billing_month.min()
print('train',train.shape,'test',test.shape,'cutoff',pd.Timestamp(cutoff).date())
print('positive rate train %.3f test %.3f'%(train.arrears_next_month.mean(),test.arrears_next_month.mean()))

## 2. Features and candidate models (named before any scoring)

In [ ]:
FEATURES=['tariff','occupants','kwh','bill_eur','late_payments_12m']   # all known on the billing date
CAT=['tariff']; NUM=[f for f in FEATURES if f not in CAT]
TARGET='arrears_next_month'
def prep(scale):
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),
                              ('n',StandardScaler() if scale else 'passthrough',NUM)])
CANDIDATES={
 'baseline_prior': Pipeline([('p',prep(False)),('m',DummyClassifier(strategy='prior'))]),
 'logreg_C0.1':    Pipeline([('p',prep(True)),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_C1':      Pipeline([('p',prep(True)),('m',LogisticRegression(C=1,max_iter=1000))]),
 'hgb_shallow':    Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=SEED))]),
 'hgb_deeper':     Pipeline([('p',prep(False)),('m',HistGradientBoostingClassifier(max_depth=6,learning_rate=0.05,max_iter=200,random_state=SEED))]),
}
print(list(CANDIDATES))

## 3. Cross-validation on training rows only (expanding window, in time order)

In [ ]:
X=train[FEATURES]; y=train[TARGET]
cv=TimeSeriesSplit(n_splits=5)          # train is sorted by month; earlier rows train, later rows validate
oof={}; rows=[]
for name,est in CANDIDATES.items():
    p=np.full(len(train),np.nan); aps=[]; aucs=[]
    for tr,va in cv.split(X):
        m=clone(est).fit(X.iloc[tr],y.iloc[tr])      # preprocessing refit inside each fold
        p[va]=m.predict_proba(X.iloc[va])[:,1]
        aps.append(average_precision_score(y.iloc[va],p[va])); aucs.append(roc_auc_score(y.iloc[va],p[va]))
    oof[name]=p
    rows.append((name,np.mean(aps),np.std(aps),np.mean(aucs)))
cvres=pd.DataFrame(rows,columns=['model','cv_pr_auc','sd','cv_roc_auc']).set_index('model').round(4)
cvres

## 4. Choose model and threshold on CV only

In [ ]:
best=cvres.cv_pr_auc.idxmax()
mask=~np.isnan(oof[best])
ths=np.linspace(0.05,0.95,91)
f1s=[f1_score(y[mask],oof[best][mask]>=t) for t in ths]
THRESH=float(ths[int(np.argmax(f1s))])
print('chosen model:',best,'| threshold (max CV F1): %.2f'%THRESH,'| CV F1 %.3f'%max(f1s))

**Choice recorded before test scoring:** the model with the highest cross-validated PR-AUC (printed above) and the decision threshold maximising out-of-fold F1. Features: tariff, occupants, kwh, bill_eur, late_payments_12m. No further tuning after this point.

## 5. Refit on all training rows; score the test set once

In [ ]:
final=clone(CANDIDATES[best]).fit(X,y)
base=clone(CANDIDATES['baseline_prior']).fit(X,y)
Xt,yt=test[FEATURES],test[TARGET]
pt=final.predict_proba(Xt)[:,1]; pb=base.predict_proba(Xt)[:,1]
pred=pt>=THRESH
print('TEST (headline) -- model:',best)
print('PR-AUC  %.3f (positive rate / no-skill = %.3f)'%(average_precision_score(yt,pt),yt.mean()))
print('ROC-AUC %.3f'%roc_auc_score(yt,pt))
print('at threshold %.2f: precision %.3f recall %.3f F1 %.3f'%(THRESH,precision_score(yt,pred),recall_score(yt,pred),f1_score(yt,pred)))
print('baseline ROC-AUC %.3f'%roc_auc_score(yt,pb))

## Conclusion
See the printed test metrics above: PR-AUC should be read against the no-skill level (the test positive rate), and ROC-AUC against 0.5. The test set contains unseen accounts in later months, so it reflects deployment. Caveat: the test is one 4-month window of a fifth of the accounts, so estimates are noisy.